In [ ]:
import pandas as pd, numpy as np
import vivarium_inputs
import gbd_mapping
import pathlib

In [ ]:
location = "India"
vehicle = "rice"

In [ ]:
location = location.title()

In [ ]:
pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

In [ ]:
children = pop[pop.index.get_level_values("age_end") <= 5].sum()
f'{int(children):,}'

In [ ]:
sim_baseline_children = pd.read_parquet(f"../../0200_pregnancy_sim/sim_results/{vehicle}/{location.lower()}/pregnancy_outcome_count.parquet")
sim_baseline_children = sim_baseline_children[
    (sim_baseline_children.scenario == 'baseline') &
    (sim_baseline_children.sub_entity == 'live_birth')
]
sim_baseline_children

In [ ]:
sim_baseline_children = sim_baseline_children.groupby("input_draw").value.sum().mean()
sim_baseline_children

In [ ]:
scalar = children / sim_baseline_children
scalar

In [ ]:
for result in ["ylds", "ylls", "deaths", "person_time"]:
    df = pd.read_parquet(f"../../0300_child_sim/sim_results/{vehicle}/{location.lower()}/{result}.parquet")
    df.value *= scalar
    path = pathlib.Path(f'../results/rescaled_child_results/{vehicle}/{location.lower()}/{result}.parquet')
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)